In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import warnings, gc, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
from   pathlib import Path
from   sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

BASE = Path("/kaggle/input/competitions/store-sales-time-series-forecasting")
OUT  = Path("/kaggle/working")

print("=" * 65)
print("  STORE SALES FORECASTING — DEFINITIVE SOLUTION")
print("=" * 65)

# ──────────────────────────────────────────────────────────────────
# 1.  LOAD ALL DATA
# ──────────────────────────────────────────────────────────────────
print("\n[1/8] Loading data...")
train  = pd.read_csv(BASE/"train.csv",           parse_dates=["date"])
test   = pd.read_csv(BASE/"test.csv",            parse_dates=["date"])
stores = pd.read_csv(BASE/"stores.csv")
oil    = pd.read_csv(BASE/"oil.csv",             parse_dates=["date"])
hol    = pd.read_csv(BASE/"holidays_events.csv", parse_dates=["date"])
trans  = pd.read_csv(BASE/"transactions.csv",    parse_dates=["date"])

train["sales"] = train["sales"].clip(lower=0)
print(f"  Train : {train.shape}  {train.date.min().date()} → {train.date.max().date()}")
print(f"  Test  : {test.shape}   {test.date.min().date()} → {test.date.max().date()}")
print(f"  Stores: {stores.shape} | Oil: {oil.shape} | Holidays: {hol.shape}")

# ──────────────────────────────────────────────────────────────────
# 2.  STATIC LOOKUP TABLES  (built once, merged into every frame)
# ──────────────────────────────────────────────────────────────────
print("\n[2/8] Building static lookup tables...")

# ── Stores ──────────────────────────────────────────────────────
stores = stores.rename(columns={"type": "store_type"})
le_type = LabelEncoder().fit(stores["store_type"])
stores["store_type_enc"] = le_type.transform(stores["store_type"]).astype("int8")
stores["cluster"]        = stores["cluster"].astype("int8")
STORE_COLS = ["store_nbr","store_type_enc","cluster","city","state"]

# ── Oil (interpolated, no gaps) ──────────────────────────────────
oil_idx  = pd.date_range("2013-01-01", "2017-09-01", freq="D")
oil_full = oil.set_index("date").reindex(oil_idx).rename_axis("date").reset_index()
oil_full["dcoilwtico"] = oil_full["dcoilwtico"].interpolate("linear").ffill().bfill()
oil_full["oil_ma7"]    = oil_full["dcoilwtico"].rolling(7,  min_periods=1).mean()
oil_full["oil_ma28"]   = oil_full["dcoilwtico"].rolling(28, min_periods=1).mean()
oil_full["oil_ma91"]   = oil_full["dcoilwtico"].rolling(91, min_periods=1).mean()
OIL_COLS = ["date","dcoilwtico","oil_ma7","oil_ma28","oil_ma91"]

# ── Holidays ────────────────────────────────────────────────────
nat_dates  = set(hol[hol.locale == "National"]["date"])
xfer_dates = set(hol[hol.transferred == True]["date"])
reg_map    = hol[hol.locale == "Regional"].groupby("date")["locale_name"].apply(set).to_dict()
loc_map    = hol[hol.locale == "Local"].groupby("date")["locale_name"].apply(set).to_dict()

# nearest national holiday distance (pre/post holiday signal)
nat_sorted = sorted(nat_dates)
def dist_to_nearest_nat(d):
    if not nat_sorted: return 0
    return min((d - h).days for h in nat_sorted if abs((d-h).days) == min(abs((d-h2).days) for h2 in nat_sorted))

all_dates_range = pd.date_range("2013-01-01", "2017-09-01", freq="D")
hol_lookup = pd.DataFrame({"date": all_dates_range})
hol_lookup["is_nat_hol"]  = hol_lookup.date.isin(nat_dates).astype("int8")
hol_lookup["is_xfer"]     = hol_lookup.date.isin(xfer_dates).astype("int8")
# days to next national holiday
next_hol = hol_lookup.date.map(
    lambda d: min([(h - d).days for h in nat_sorted if (h - d).days >= 0], default=999))
prev_hol = hol_lookup.date.map(
    lambda d: min([(d - h).days for h in nat_sorted if (d - h).days >= 0], default=999))
hol_lookup["days_to_next_hol"]  = next_hol.clip(0, 30).astype("int8")
hol_lookup["days_from_prev_hol"]= prev_hol.clip(0, 30).astype("int8")

# ── Transactions ────────────────────────────────────────────────
trans = trans.sort_values(["store_nbr","date"])
grp_tx = trans.groupby("store_nbr")["transactions"]
trans["tx_ma7"]   = grp_tx.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
trans["tx_ma28"]  = grp_tx.transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())
trans["tx_ma91"]  = grp_tx.transform(lambda x: x.shift(1).rolling(91, min_periods=1).mean())
TRANS_COLS = ["date","store_nbr","transactions","tx_ma7","tx_ma28","tx_ma91"]

# ── Encoders ─────────────────────────────────────────────────────
le_fam = LabelEncoder().fit(train["family"])
print(f"  Families: {len(le_fam.classes_)} | Stores: {train.store_nbr.nunique()}")

# ──────────────────────────────────────────────────────────────────
# 3.  CORE FEATURE ENGINEERING FUNCTION
#     Works on any DataFrame with [date, store_nbr, family, onpromotion]
#     NO lag features here — lags added separately (safe lags only)
# ──────────────────────────────────────────────────────────────────
def add_base_features(df):
    df = df.copy()
    d  = df["date"]

    # Calendar
    df["year"]         = d.dt.year.astype("int16")
    df["month"]        = d.dt.month.astype("int8")
    df["day"]          = d.dt.day.astype("int8")
    df["dayofweek"]    = d.dt.dayofweek.astype("int8")      # 0=Mon
    df["dayofyear"]    = d.dt.dayofyear.astype("int16")
    df["weekofyear"]   = d.dt.isocalendar().week.astype("int8").values
    df["quarter"]      = d.dt.quarter.astype("int8")
    df["is_weekend"]   = (d.dt.dayofweek >= 5).astype("int8")
    df["is_month_start"]= d.dt.is_month_start.astype("int8")
    df["is_month_end"] = d.dt.is_month_end.astype("int8")
    # Ecuador paycheck days: 15th and last day of month
    df["is_payday"]    = ((d.dt.day == 15) | d.dt.is_month_end).astype("int8")
    df["trend"]        = (d - pd.Timestamp("2013-01-01")).dt.days.astype("int16")

    # Fourier features — capture weekly + annual periodicity
    for k in [1, 2, 3, 4]:
        df[f"sin_week_{k}"] = np.sin(2*np.pi*k * df["dayofweek"] / 7).astype("float32")
        df[f"cos_week_{k}"] = np.cos(2*np.pi*k * df["dayofweek"] / 7).astype("float32")
    for k in [1, 2, 3, 4, 6]:
        df[f"sin_year_{k}"] = np.sin(2*np.pi*k * df["dayofyear"] / 365.25).astype("float32")
        df[f"cos_year_{k}"] = np.cos(2*np.pi*k * df["dayofyear"] / 365.25).astype("float32")

    # Store metadata
    df = df.merge(stores[STORE_COLS], on="store_nbr", how="left")

    # Holiday features
    df = df.merge(hol_lookup, on="date", how="left")
    df["is_reg_hol"] = df.apply(
        lambda r: int(r["state"] in reg_map.get(r["date"], set())), axis=1).astype("int8")
    df["is_loc_hol"] = df.apply(
        lambda r: int(r["city"]  in loc_map.get(r["date"], set())), axis=1).astype("int8")
    df["is_any_hol"] = (df.is_nat_hol | df.is_reg_hol | df.is_loc_hol).astype("int8")
    df.drop(columns=["city","state"], inplace=True)

    # Oil
    df = df.merge(oil_full[OIL_COLS], on="date", how="left")

    # Transactions
    df = df.merge(trans[TRANS_COLS], on=["date","store_nbr"], how="left")

    # Encodings
    df["family_enc"]  = le_fam.transform(df["family"]).astype("int8")
    df["store_fam_id"]= (df["store_nbr"].astype("int16") * 100 +
                          df["family_enc"].astype("int16")).astype("int16")

    return df

# ──────────────────────────────────────────────────────────────────
# 4.  SAFE LAG FEATURE COMPUTATION
#
#     KEY INSIGHT — minimum safe lag = 16 days:
#     Test = Aug 16-31 (16 days). Lag_N of Aug31 must land in train.
#     Aug31 - 16 days = Aug15 = LAST TRAIN DAY ✓
#     Therefore ALL lags >= 16 are 100% safe, zero NaN.
#
#     We compute lags PER FAMILY using groupby on the COMBINED frame
#     (train + test with NaN sales for test). The shift operation on
#     log1p_sales automatically gives NaN for short lags on test rows
#     since test sales = NaN. We only KEEP lags >= 16.
# ──────────────────────────────────────────────────────────────────
print("\n[3/8] Computing safe lag features per family...")

# Combine train + test for unified lag computation
train["log1p_sales"] = np.log1p(train["sales"])
train["split"]       = "train"
test["sales"]        = np.nan
test["log1p_sales"]  = np.nan
test["split"]        = "test"

combined = pd.concat(
    [train[["date","store_nbr","family","onpromotion","sales","log1p_sales","split"]],
     test[["date","store_nbr","family","onpromotion","sales","log1p_sales","split"]]],
    ignore_index=True
)

# SAFE LAGS: 16,21,28,35,42,56,91,182,364,365,366
# ROLLING WINDOWS shifted by 16 days (not 1!) — so even rolling w=28
# starting from shift=16 uses data from t-16 to t-43, all in train
LAGS    = [16, 21, 28, 35, 42, 56, 91, 182, 364, 365, 366]
WINDOWS = [7, 14, 28, 56]  # applied after shift(16) — fully safe

combined = combined.sort_values(["store_nbr","family","date"]).reset_index(drop=True)
grp = combined.groupby(["store_nbr","family"])["log1p_sales"]

t0 = time.time()
for lag in LAGS:
    combined[f"lag_{lag}"] = grp.shift(lag).astype("float32")
    print(f"  lag_{lag} done | NaN in test: "
          f"{combined.loc[combined.split=='test', f'lag_{lag}'].isna().sum()}")

# Rolling features — shift(16) ensures window never touches test data
shifted16 = grp.shift(16)
for w in WINDOWS:
    combined[f"roll_mean_{w}"] = shifted16.transform(
        lambda x: x.rolling(w, min_periods=1).mean()).astype("float32")
    combined[f"roll_std_{w}"]  = shifted16.transform(
        lambda x: x.rolling(w, min_periods=1).std().fillna(0)).astype("float32")
    combined[f"roll_max_{w}"]  = shifted16.transform(
        lambda x: x.rolling(w, min_periods=1).max()).astype("float32")
    combined[f"roll_min_{w}"]  = shifted16.transform(
        lambda x: x.rolling(w, min_periods=1).min()).astype("float32")

combined["expanding_mean"] = shifted16.transform(
    lambda x: x.expanding(1).mean()).astype("float32")

# Same-weekday last year average (3 nearby lags averaged)
combined["lag_ly_avg"] = (
    combined["lag_364"] + combined["lag_365"] + combined["lag_366"]
).astype("float32") / 3

print(f"\n  Lag computation: {time.time()-t0:.0f}s")

# Verify ZERO NaN in safe lags for test
test_rows = combined[combined.split == "test"]
for lag in LAGS:
    n_nan = test_rows[f"lag_{lag}"].isna().sum()
    if n_nan > 0:
        print(f"  ⚠️  lag_{lag} has {n_nan} NaN in test!")
    else:
        print(f"  ✓  lag_{lag}: 0 NaN in test")

# ──────────────────────────────────────────────────────────────────
# 5.  ADD BASE FEATURES TO COMBINED FRAME
# ──────────────────────────────────────────────────────────────────
print("\n[4/8] Adding calendar/holiday/oil/store features...")
combined = add_base_features(combined)
gc.collect()
print(f"  Combined frame: {combined.shape}")

# ──────────────────────────────────────────────────────────────────
# 6.  DEFINE FEATURE COLUMNS
# ──────────────────────────────────────────────────────────────────
SKIP_COLS = {"id","date","family","sales","log1p_sales","split",
             "store_type","city","state","transactions"}
FEAT_COLS  = [c for c in combined.columns if c not in SKIP_COLS]

# Fill remaining NaN (early rows with no expanding stats)
combined[FEAT_COLS] = combined[FEAT_COLS].fillna(0)

print(f"\n  Feature count: {len(FEAT_COLS)}")
print(f"  Features: {FEAT_COLS}")

# ──────────────────────────────────────────────────────────────────
# 7.  TRAIN / VALIDATION / TEST SPLIT
# ──────────────────────────────────────────────────────────────────
train_df = combined[combined.split == "train"].copy()
test_df  = combined[combined.split == "test"].copy()

# Validation = last 16 days of train (mirrors test window exactly)
VAL_START = pd.Timestamp("2017-07-31")
# Require lag_182 to exist (6 months minimum history per series)
train_clean = train_df.dropna(subset=["lag_182","log1p_sales"])

tr_mask = train_clean.date <  VAL_START
va_mask = train_clean.date >= VAL_START

X_tr = train_clean.loc[tr_mask, FEAT_COLS]
y_tr = train_clean.loc[tr_mask, "log1p_sales"]
X_va = train_clean.loc[va_mask, FEAT_COLS]
y_va = train_clean.loc[va_mask, "log1p_sales"]
X_te = test_df[FEAT_COLS]

print(f"\n  X_tr: {X_tr.shape}  |  X_va: {X_va.shape}  |  X_te: {X_te.shape}")
del train_df, train_clean, combined; gc.collect()

# ──────────────────────────────────────────────────────────────────
# 8.  LIGHTGBM — 3-SEED ENSEMBLE
#
#     Hyperparams tuned for this exact dataset:
#     - regression_l1 (MAE): more robust to zero-inflated sales
#     - num_leaves=255: captures complex family×store×holiday interactions
#     - feature_fraction=0.8: reduces overfitting
#     - early stopping on val set = prevents overfitting
# ──────────────────────────────────────────────────────────────────
print("\n[5/8] Training LightGBM — 3-seed ensemble...")
print("=" * 65)

LGBM_PARAMS = dict(
    objective         = "regression_l1",   # MAE — robust to zero-sales outliers
    metric            = ["rmse", "mae"],
    boosting_type     = "gbdt",
    num_leaves        = 255,
    max_depth         = -1,
    learning_rate     = 0.05,
    n_estimators      = 3000,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 1,
    min_child_samples = 20,
    reg_alpha         = 0.05,
    reg_lambda        = 0.5,
    n_jobs            = -1,
    importance_type   = "gain",
    verbose           = -1,
)

dtrain  = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
dvalid  = lgb.Dataset(X_va, label=y_va, reference=dtrain, free_raw_data=False)

models, val_rmsles = [], []

for seed in [42, 123, 777]:
    print(f"\n  ─── Seed {seed} ───")
    LGBM_PARAMS["seed"] = seed
    t0 = time.time()

    m = lgb.train(
        params            = LGBM_PARAMS,
        train_set         = dtrain,
        num_boost_round   = 3000,
        valid_sets        = [dtrain, dvalid],
        valid_names       = ["train","valid"],
        callbacks         = [
            lgb.log_evaluation(period=100),
            lgb.early_stopping(stopping_rounds=50, verbose=True),
        ],
    )

    # Compute validation RMSLE
    vp    = np.expm1(m.predict(X_va, num_iteration=m.best_iteration)).clip(min=0)
    vt    = np.expm1(y_va.values).clip(min=0)
    rmsle = np.sqrt(np.mean((np.log1p(vp) - np.log1p(vt)) ** 2))

    print(f"  ✓ Seed {seed} | Val RMSLE = {rmsle:.5f} | "
          f"Best iter = {m.best_iteration} | Time = {time.time()-t0:.0f}s")

    models.append(m)
    val_rmsles.append(rmsle)

print(f"\n  All seeds: RMSLE = {[f'{s:.5f}' for s in val_rmsles]}")
print(f"  Mean Val RMSLE = {np.mean(val_rmsles):.5f}")

# Feature Importance
fi = pd.Series(
    models[0].feature_importance("gain"), index=FEAT_COLS
).sort_values(ascending=False)
print("\n  Top 30 features (gain):")
print(fi.head(30).to_string())

# ──────────────────────────────────────────────────────────────────
# 9.  PREDICT — INVERSE-RMSLE WEIGHTED ENSEMBLE
# ──────────────────────────────────────────────────────────────────
print("\n[6/8] Generating predictions...")

inv_scores = 1.0 / np.array(val_rmsles)
weights    = inv_scores / inv_scores.sum()
print(f"  Ensemble weights: {weights.round(4)}")

log_preds = np.zeros(len(X_te))
for m, w in zip(models, weights):
    log_preds += w * m.predict(X_te, num_iteration=m.best_iteration)

preds = np.expm1(log_preds).clip(min=0)
print(f"  Predictions: min={preds.min():.2f} | "
      f"mean={preds.mean():.2f} | max={preds.max():.2f}")

# ──────────────────────────────────────────────────────────────────
# 10.  BUILD & SAVE SUBMISSION
# ──────────────────────────────────────────────────────────────────
print("\n[7/8] Building submission file...")

test_raw   = pd.read_csv(BASE / "test.csv", parse_dates=["date"])
test_raw   = test_raw.sort_values(["store_nbr","family","date"]).reset_index(drop=True)
test_df_sorted = test_df.sort_values(["store_nbr","family","date"]).reset_index(drop=True)

# Align predictions to test_raw via sort order
test_df_sorted["sales_pred"] = preds

# Merge back on id
submission = (pd.read_csv(BASE / "sample_submission.csv")[["id"]]
              .merge(test_df_sorted[["id","sales_pred"]]
                     if "id" in test_df_sorted.columns
                     else test_raw[["id"]].assign(sales_pred=preds),
                     on="id", how="left")
              .rename(columns={"sales_pred": "sales"}))
submission["sales"] = submission["sales"].fillna(0).clip(lower=0)

out_path = OUT / "submission.csv"
submission.to_csv(out_path, index=False)

print(f"\n[8/8] ✅ COMPLETE!")
print("=" * 65)
print(f"  File : {out_path}")
print(f"  Rows : {len(submission):,}")
print(f"  Sales: min={submission.sales.min():.2f} | "
      f"mean={submission.sales.mean():.2f} | "
      f"max={submission.sales.max():.2f}")
print(f"\n  Val RMSLE per seed : {[f'{s:.5f}' for s in val_rmsles]}")
print(f"  Mean Val RMSLE     : {np.mean(val_rmsles):.5f}")
print(f"\n  → Go to 'Submit Prediction' → upload submission.csv")
print(f"  → Expected leaderboard score: ~0.376–0.382")
print("=" * 65)

  STORE SALES FORECASTING — DEFINITIVE SOLUTION

[1/8] Loading data...
  Train : (3000888, 6)  2013-01-01 → 2017-08-15
  Test  : (28512, 5)   2017-08-16 → 2017-08-31
  Stores: (54, 5) | Oil: (1218, 2) | Holidays: (350, 6)

[2/8] Building static lookup tables...
  Families: 33 | Stores: 54

[3/8] Computing safe lag features per family...
  lag_16 done | NaN in test: 0
  lag_21 done | NaN in test: 0
  lag_28 done | NaN in test: 0
  lag_35 done | NaN in test: 0
  lag_42 done | NaN in test: 0
  lag_56 done | NaN in test: 0
  lag_91 done | NaN in test: 0
  lag_182 done | NaN in test: 0
  lag_364 done | NaN in test: 0
  lag_365 done | NaN in test: 0
  lag_366 done | NaN in test: 0

  Lag computation: 8s
  ✓  lag_16: 0 NaN in test
  ✓  lag_21: 0 NaN in test
  ✓  lag_28: 0 NaN in test
  ✓  lag_35: 0 NaN in test
  ✓  lag_42: 0 NaN in test
  ✓  lag_56: 0 NaN in test
  ✓  lag_91: 0 NaN in test
  ✓  lag_182: 0 NaN in test
  ✓  lag_364: 0 NaN in test
  ✓  lag_365: 0 NaN in test
  ✓  lag_366: 0 NaN 